# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
bks = spark.table('bookings')
mems = spark.table('members')
fac = spark.table('facilities')

In [0]:
# Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

from pyspark.sql import functions as F

result = (
    bks.filter(
        (bks["starttime"] >= "2012-09-01") & 
        (bks["starttime"] < "2012-10-01")
    )
    .groupBy("facid")
    .agg(F.sum("slots").alias("Total Slots"))
    .orderBy("Total Slots")
)

# Save the DataFrame to a Parquet file
result.write.mode("overwrite").parquet("/Volumes/jarvis-pyspark/landing/jarvis-volume")


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
result = (
    mems.alias("mems")
    .join(
        bks.alias("bks"), 
        mems["memid"] == bks["memid"], 
        "inner"
    )
    .join(
        fac.alias("facs"), 
        bks["facid"] == fac["facid"], 
        "inner"
    )
    .filter(fac["name"].isin("Tennis Court 1", "Tennis Court 2"))
    .select(
        F.concat_ws(" ", mems["firstname"], mems["surname"]).alias("member"),
        fac["name"].alias("facility")
    )
    .distinct()
    .orderBy("member", "facility")
)

# Save the partitioned DataFrame as a managed Delta table
(
    result.write
    .mode("overwrite")
    .format("delta")
    .partitionBy("facility")
    .saveAsTable("threejoin_delta")
)

## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
API_KEY = "W14XMA0VUYM4BDFP"
url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&outputsize=compact&datatype=json&apikey={API_KEY}"
response = requests.get(url, timeout=10)
data = response.json()
print(data)

{'Information': 'We have detected your API key as W14XMA0VUYM4BDFP and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.'}


In [0]:
import requests
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, MapType

API_KEY = "MEGPLOTQVE49UBAW"
quote_tickers = ["GOOGL", "AAPL", "MSFT", "TSLA"]

def get_quote_json(symbol):
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&datatype=json&apikey={API_KEY}"
    response = requests.get(url, timeout=10)
    return response.text

# 1. Collect all raw JSON records into a list of DataFrames first
df_list = []
for ticker in quote_tickers:
    raw_json = get_quote_json(ticker)
    
    # Create the single row DataFrame matching your landing strategy
    company_df = spark.createDataFrame(
        [(ticker, raw_json)],
        ["symbol", "raw_json"]
    )
    df_list.append(company_df)

# 2. Union them all safely (no conflicting schema fields or names yet)
full_raw_df = reduce(lambda df1, df2: df1.union(df2), df_list)

# 3. Define mapping schema for parsing
json_schema = StructType([
    StructField("Meta Data", MapType(StringType(), StringType()), True),
    StructField("Time Series (Daily)", MapType(StringType(), MapType(StringType(), StringType())), True)
])

# 4. Parse the combined dataset
parsed_df = (
    full_raw_df.withColumn("parsed", F.from_json(full_raw_df["raw_json"], json_schema))
    .select(
        full_raw_df["symbol"],
        F.explode("parsed.Time Series (Daily)").alias("date", "metrics")
    )
    .select(
        "symbol",
        "date",
        F.col("metrics")["4. close"].cast("double").alias("close_price")
    )
)

# 5. Extract year and week natively to bypass the Spark 3.0 parsing error completely
result_df = (
    parsed_df.withColumn("date_obj", F.to_date(parsed_df["date"]))
    .withColumn("year", F.year("date_obj"))
    .withColumn("week", F.weekofyear("date_obj"))
    .groupBy("symbol", "year", "week")
    .agg(F.max("close_price").alias("max_closing_price"))
    .orderBy("symbol", "year", "week")
)

# 6. Save the table partitioned by symbol
(
    result_df.write
    .mode("overwrite")
    .format("delta")
    .partitionBy("symbol")
    .saveAsTable("max_closing_price_weekly")
)

result_df.show()

+------+----+----+-----------------+
|symbol|year|week|max_closing_price|
+------+----+----+-----------------+
| GOOGL|2026|   8|           314.98|
| GOOGL|2026|   9|            312.9|
| GOOGL|2026|  10|           306.52|
| GOOGL|2026|  11|            308.7|
| GOOGL|2026|  12|           310.92|
| GOOGL|2026|  13|           302.06|
| GOOGL|2026|  14|           297.39|
| GOOGL|2026|  15|           318.49|
| GOOGL|2026|  16|           341.68|
| GOOGL|2026|  17|            344.4|
| GOOGL|2026|  18|           385.69|
| GOOGL|2026|  19|            400.8|
| GOOGL|2026|  20|           402.62|
| GOOGL|2026|  21|           396.94|
| GOOGL|2026|  22|           390.13|
| GOOGL|2026|  23|           376.37|
| GOOGL|2026|  24|           364.26|
| GOOGL|2026|  25|           373.25|
| GOOGL|2026|  26|           349.68|
| GOOGL|2026|  27|           361.21|
+------+----+----+-----------------+
only showing top 20 rows


In [0]:
%sql
SELECT * FROM max_closing_price_weekly;

symbol,year,week,max_closing_price
GOOGL,2026,24,364.26
GOOGL,2026,20,402.62
GOOGL,2026,12,310.92
GOOGL,2026,28,367.03
GOOGL,2026,9,312.9
GOOGL,2026,23,376.37
GOOGL,2026,22,390.13
GOOGL,2026,25,373.25
GOOGL,2026,18,385.69
GOOGL,2026,27,361.21


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# 1. Connection configuration using official RNAcentral credentials
jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"
properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

# 2. Extract 100 records using a subquery to push down the LIMIT directly to PostgreSQL
# (Note: The subquery must be wrapped in parentheses and given an alias in Spark)
query_pushdown = "(SELECT * FROM rna LIMIT 100) AS rna_subset"

rna_df = spark.read.jdbc(
    url=jdbc_url, 
    table=query_pushdown, 
    properties=properties
)

# 3. Transform 
# Per your requirements, loading as-is with no modifications required.

# 4. Load into a managed Delta table
(
    rna_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("rna_100_records")
)

# Preview the loaded data
rna_df.show(5)


+-------+-------------+-------------------+---------+----------------+---+--------------------+--------+--------------------+
|     id|          upi|          timestamp|userstamp|           crc64|len|           seq_short|seq_long|                 md5|
+-------+-------------+-------------------+---------+----------------+---+--------------------+--------+--------------------+
|9941908|URS000097B394|2016-03-16 17:23:10|   RNACEN|45FE7CFC916229FF|820|GGTCAAGCTACTAAGGG...|    NULL|d4f3dd62e6b033718...|
|9941912|URS000097B398|2016-03-16 17:23:10|   RNACEN|BB5FC13A87E8BDEA|512|CGGGGAAAGAAGATCCT...|    NULL|d4f507918aa15409f...|
|9941913|URS000097B399|2016-03-16 17:23:10|   RNACEN|19E718DFEACE3446| 57|TGACCTCAGATCAAGTA...|    NULL|d4f54ba099e046e5a...|
|9941918|URS000097B39E|2016-03-16 17:23:10|   RNACEN|41E8503457E6A42F|151|TACCGATTGAATGATCC...|    NULL|3f0ee5411c427fc90...|
|9941919|URS000097B39F|2016-03-16 17:23:10|   RNACEN|3AA634D58CA826A3|897|CGATGAAGGACGTGGTA...|    NULL|d4f841c9079b97